In [2]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.utils import resample
from scipy.optimize import minimize
from scipy.stats import norm
from scipy.stats import chi2
import statsmodels.api as sm

In [4]:
# Read file
df = pd.read_excel('Assignment0Data.xlsx', sheet_name = 'StocksBondsBills')

# Create datetime and set as index
df['Date'] = pd.to_datetime(df[['Year', 'Month']].assign(DAY = 1))
df.set_index('Date', inplace=True)


# Save file for results
output_file = 'Assignment0_results.xlsx'

In [3]:
# 1a)
# Summary statistics for stocks, bonds, and bills
# Save to result file

# Calculate mean, stdev, variance, skewness, and kurtosis for return series
df_summary_statistics = df.agg(
                                    {
                                        "Stocks": ["mean", "std", "var", "skew", "kurt"],
                                        "Bonds": ["mean", "std", "var", "skew", "kurt"],
                                        "Bills": ["mean", "std", "var", "skew", "kurt"]
                                    }
                                )

# Save to result file
with pd.ExcelWriter(output_file, engine='openpyxl', mode='a' if os.path.exists(output_file) else 'w', if_sheet_exists='replace') as writer:
    sheet_name = 'summary_statistics'
    df_summary_statistics.to_excel(writer, sheet_name=sheet_name)

In [ ]:
# 1a)
# Plot excess stock returns
# Calculate excess returns of stocks
df['Excess_stocks'] = df['Stocks'] - df['Bills']

# Plot figure
plt.figure(figsize=(12,8))
plt.plot(df.index, df['Excess_stocks'], label = 'Excess Stock Returns', color = 'g')
plt.axhline(y = 0, color = 'k', linewidth = 2, linestyle = '--')
plt.title('Excess Stock Returns')
plt.xlabel('Date')
plt.ylabel('Excess Returns')
plt.legend()
plt.grid(True)

# Save figure
plt.savefig('Assignment0_excess_returns_plot.jpg', format = 'jpg', dpi = 300)



In [5]:
# 1b)
# Number of observations
num_obs = len(df['Excess_stocks'])

# Point estimates for mean return and variance
mean_return = df['Excess_stocks'].mean()
variance_returns = df['Excess_stocks'].var()

# Standard deviation
sigma_returns = df['Excess_stocks'].std()

# Standard error of mean return
standard_error_return = sigma_returns / np.sqrt(num_obs)

# Standard error of variance
standard_error_variance = np.sqrt((2 * variance_returns ** 2) / (num_obs - 1))

# Store in df
results = pd.DataFrame({
    'Statistic': ['Market Risk Premium', 'Variance', 'Standard Error of Premium', 'Standard Error of Variance'],
    'Value': [mean_return, variance_returns, standard_error_return, standard_error_variance]
})

# Save to file
with pd.ExcelWriter(output_file, engine='openpyxl', mode='a' if os.path.exists(output_file) else 'w', if_sheet_exists='replace') as writer:
    results.to_excel(writer, sheet_name='1b)Market_premium', index=False)


In [6]:
# 1c)

# Analytical
# Sharpe Ratio
sharpe_ratio = mean_return / sigma_returns

# Variance of mean returns
var_mean_returns = sigma_returns ** 2 / num_obs
# Variance of sigma
var_sigma = sigma_returns ** 2 / (2 * (num_obs - 1))

# For the delta method, we calculate partial derivatives for the sharpe ratio
partial_sharpe_returns = 1 / sigma_returns
partial_sharpe_sigma = - mean_return / (sigma_returns ** 2)

# Apply delta method to estimate variance of sharpe ratio
variance_sharpe = (partial_sharpe_returns ** 2 * var_mean_returns) + (partial_sharpe_sigma ** 2 * var_sigma)

# Standard error of sharpe ratio
se_sharpe = np.sqrt(variance_sharpe)

# Numerically 
# We instead estimate derivatives by adding a small epsilon to returns and sigma respectively
epsilon = 0.0000001

# Estimate partial derivates with epsilon
partial_sharpe_returns_numerical = (((mean_return + epsilon) / sigma_returns) - (mean_return / sigma_returns)) / epsilon
partial_sharpe_sigma_numerical = ((mean_return / (sigma_returns + epsilon)) - (mean_return / sigma_returns)) / epsilon

# Delta with numerical derivatives
variance_sharpe_numerical = (partial_sharpe_returns_numerical ** 2 * var_mean_returns) + (partial_sharpe_sigma_numerical ** 2 * var_sigma)
se_sharpe_numerical = np.sqrt(variance_sharpe_numerical)

# Save results
results_sharpe = pd.DataFrame({
    'Statistic':['Sharpe Ratio','Variance Mean returns' 'Variance Sharpe', 'SE Sharpe', 'Variance Sharpe Numerical', 'SE Sharpe Numerical'],
    'Value': [sharpe_ratio, variance_sharpe, se_sharpe, variance_sharpe_numerical, se_sharpe_numerical]
})

with pd.ExcelWriter(output_file, engine='openpyxl', mode='a' if os.path.exists(output_file) else 'w', if_sheet_exists='replace') as writer:
    results_sharpe.to_excel(writer, sheet_name='1c)Sharpe_se', index=False)



In [15]:
# 1d)
# Set up bootstrap
num_iterations = 10000

# Length of sample
T = len(df['Excess_stocks'])

# Store bootstrapped sharpe ratios
bootstrapped_sharpe_ratios =[]

# Bootstrap. We first resample with replacements. Then calculate mean and sigma of the current sample, and finally calculate sharpe ratio and store the value.
for i in range(num_iterations):
    current_sample = resample(df['Excess_stocks'], n_samples = T, replace = True)

    mean_current_sample = current_sample.mean()
    sigma_current_sample = current_sample.std()

    sharpe_current_sample = mean_current_sample / sigma_current_sample
    bootstrapped_sharpe_ratios.append(sharpe_current_sample)


# SE of bootstrapped sharpe ratios
bootstrapped_sharpe_ratios = np.array(bootstrapped_sharpe_ratios)
se_sharpe_bootstrapped = np.std(bootstrapped_sharpe_ratios)

mean_bootstrap_sharpe = bootstrapped_sharpe_ratios.mean()

# Confidence intervall
confidence_intervall_95 = np.percentile(bootstrapped_sharpe_ratios, [2.5, 97.5])

# Save results
results_bootstrap = pd.DataFrame({
    'Statistic': ['Mean Bootstrapped Sharpe', 'SE of Sharpe (Bootstrapped)', '95% CI Lower Bound', '95% CI Upper Bound'],
    'Value': [mean_bootstrap_sharpe, se_sharpe_bootstrapped, confidence_intervall_95[0], confidence_intervall_95[1]]
})

with pd.ExcelWriter(output_file, engine='openpyxl', mode='a' if os.path.exists(output_file) else 'w', if_sheet_exists='replace') as writer:
    results_bootstrap.to_excel(writer, sheet_name='1d)Bootstrapped_SE_CI', index=False)


In [16]:
# 2)

# Create data. 100 observations with mean = 1 and stdev = 2
np.random.seed(123)
data = np.random.normal(loc = 1, scale = 2, size = 100)

# Check the mean and stdev of the simulated data as a control
# print(data.mean()) # Mean of simulated data of about 1.05 for this seed
# print(data.std())  # Stdev for simulated data of about 2.25 for this seed

# Set up log-likelihood function. Return the negative of the function as scipy's optimization can only minimize
def log_likelihood(parameters, data):
    mean, sigma = parameters
    n = len(data)
    function_value = -0.5 * n * np.log(2 * np.pi) - n * np.log(sigma) - (1 / (2 * sigma ** 2)) * np.sum((data - mean) ** 2)
    return -function_value

# Find max of log-likelihood with scipy's minimize function         Note: Results are robust to varying initial guesses. 
initial_guess = [0, 2]

# Set bounds on mean and sigma. Force sigma to be positive
bounds = [(-np.inf, np.inf), (0.0000001, np.inf)]

result = minimize(fun = log_likelihood, x0 = initial_guess, args = (data, ), bounds = bounds)

# Save estimated parameters 
mean_hat, sigma_hat = result.x

# Calculate mean log-likelihood value
log_likelihood_value = -log_likelihood([mean_hat, sigma_hat], data)
mean_log_likelihood = log_likelihood_value / len(data)


# Standard Errors
# The scipy optimization automatically calculates the hessian matrix. We grab the inverse of the Hessian matrix as an approximate for the variance-covariance matrix. 
inverse_hessian = result.hess_inv.todense()

# Take the square root of the diagonal (variances) to calculate standard errors
se_mean, se_sigma = np.sqrt(np.diag(inverse_hessian))

# Save results

results_loglikelihood = pd.DataFrame({
    'Statistic': ['Estimated Mean', 'Estimated Sigma', 'Mean Log-Likelihood', 'SE of Mean', 'SE of Sigma'],
    'Value': [mean_hat, sigma_hat, mean_log_likelihood, se_mean, se_sigma]
})

with pd.ExcelWriter(output_file, engine='openpyxl', mode='a' if os.path.exists(output_file) else 'w', if_sheet_exists='replace') as writer:
    results_loglikelihood.to_excel(writer, sheet_name='2)LogLikelihood', index=False)


In [ ]:
# 2) cont.
# Create ranges for different mean and sigmas. 
mean_range = np.linspace(mean_hat - 0.5, mean_hat + 0.5, 20)
sigma_range = np.linspace(sigma_hat - 0.5, sigma_hat + 0.5, 20)
Mean, Sigma = np.meshgrid(mean_range, sigma_range)
LogL = np.zeros_like(Mean)

# Calculate value for log-likelihood for each combination of mean and sigma
for i in range(Mean.shape[0]):
    for j in range(Mean.shape[1]):
        LogL[i, j] = -log_likelihood([Mean[i, j], Sigma[i, j]], data)

# Countour plot
plt.figure(figsize=(12, 8))
contour = plt.contour(Mean, Sigma, LogL, levels=20, cmap='viridis')
plt.clabel(contour, inline=True, fontsize=8)
plt.scatter(mean_hat, sigma_hat, color='red', marker='x', label='Estimated Parameters') # Estimated parameters
plt.scatter(1, 2, color='red', marker='o', label='True Parameters') # True mean and sigma
plt.xlabel('Mean')
plt.ylabel('Sigma')
plt.title('Log-Likelihood Contour Plot')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('Assignment0_countour_plot_logL.jpg', format = 'jpg', dpi = 300)
plt.show()


In [5]:
# 3)

# Read data
df_fama = pd.read_excel('Assignment0Data.xlsx', sheet_name = 'FamaFrench5Factors')

# Create datetime and set as index
df_fama['Date'] = pd.to_datetime(df_fama[['Year', 'Month']].assign(DAY = 1))
df_fama.set_index('Date', inplace=True)

# 3a) 

# Calculate mean, stdev, variance, skewness, and kurtosis for return series
df_fama_summary_stat = df_fama.agg(
                                    {
                                        "Mkt-RF": ["mean", "std", "var", "skew", "kurt"],
                                        "SMB": ["mean", "std", "var", "skew", "kurt"],
                                        "HML": ["mean", "std", "var", "skew", "kurt"],
                                        "RMW": ["mean", "std", "var", "skew", "kurt"],
                                        "CMA": ["mean", "std", "var", "skew", "kurt"],
                                    }
                                )

# Save to result file
with pd.ExcelWriter(output_file, engine='openpyxl', mode='a' if os.path.exists(output_file) else 'w', if_sheet_exists='replace') as writer:
    sheet_name = '3a)Summary_statistics'
    df_fama_summary_stat.to_excel(writer, sheet_name=sheet_name)


In [10]:
# 3b)

# SMB Regression
x = df_fama['Mkt-RF']
x = sm.add_constant(x)
y1 = df_fama['SMB']

model_smb = sm.OLS(y1, x).fit()
#print(model_smb.summary())

# HML Regression
y2 = df_fama['HML']

model_hml = sm.OLS(y2, x).fit()
#print(model_hml.summary())

# Save OLS parameters
alpha_smb, beta_smb = model_smb.params
alpha_hml, beta_hml = model_hml.params

# OLS SE homoskedastic
se_ols_smb = model_smb.bse
se_ols_hml = model_hml.bse

# Heteroskedasticity-consistent SE (White 1980)
se_hc_ols_smb = model_smb.HC0_se
se_hc_ols_hml = model_hml.HC0_se

# T-values
tvalue_alpha_smb, tvalue_beta_smb = model_smb.tvalues
tvalue_alpha_hml, tvalue_beta_hml = model_hml.tvalues

# P-values
pvalue_alpha_smb, pvalue_beta_smb = model_smb.pvalues
pvalue_alpha_hml, pvalue_beta_hml = model_hml.pvalues

# Save results
model_smb_summary = pd.DataFrame({
    'Coefficient': model_smb.params,
    'Standard_Error': model_smb.bse,
    't-value': model_smb.tvalues,
    'p-value': model_smb.pvalues
})

model_hml_summary = pd.DataFrame({
    'Coefficient': model_hml.params,
    'Standard_Error': model_hml.bse,
    't-value': model_hml.tvalues,
    'p-value': model_hml.pvalues
})

results_3b = pd.DataFrame({
    'Statistic': ['Alpha_SMB', 'Beta_SMB', 'Alpha_HML', 'Beta_HML', 'OLS SE Alpha_SMB', 'White SE Alpha_SMB', 
                  'OLS SE Alpha_HML', 'White SE Alpha_HML', 'OLS SE Beta_SMB', 'White SE Beta_SMB', 
                  'OLS SE Beta_HML', 'White SE Beta_HML', 't-Statistic Alpha_SMB', 'p-value Alpha_SMB', 
                  't-Statistic Alpha_HML', 'p-value Alpha_HML'],
    'Value': [alpha_smb, beta_smb, alpha_hml, beta_hml, se_ols_smb['const'], se_hc_ols_smb['const'], 
              se_ols_hml['const'], se_hc_ols_hml['const'],
              se_ols_smb['Mkt-RF'], se_hc_ols_smb['Mkt-RF'], 
              se_ols_hml['Mkt-RF'], se_hc_ols_hml['Mkt-RF'], 
              tvalue_alpha_smb, pvalue_alpha_smb, 
              tvalue_alpha_hml, pvalue_alpha_hml]})


with pd.ExcelWriter(output_file, engine='openpyxl', mode='a' if os.path.exists(output_file) else 'w', if_sheet_exists='replace') as writer:
    model_smb_summary.to_excel(writer, sheet_name = '3b)Regression_SMB_MktRF')
    model_hml_summary.to_excel(writer, sheet_name = '3b)Regression_HML_MktRF')
    results_3b.to_excel(writer, sheet_name = '3b)Results', index = False)



In [49]:
# 3c)
# Extract covariances
cov_smb = model_smb.cov_params()
cov_hml = model_hml.cov_params()

# Test alpha_smb = 0
wald_alpha_smb = (alpha_smb - 0) ** 2 / cov_smb.loc['const', 'const']
p_value_alpha_smb = 1 - chi2.cdf(wald_alpha_smb, df = 1)

# Test alpha_hml = 0
wald_alpha_hml = (alpha_hml - 0) ** 2 / cov_hml.loc['const', 'const']
p_value_alpha_hml = 1 - chi2.cdf(wald_alpha_hml, df = 1)

# Test alpha_smb = alpha_hml = 0 
vector = np.array([alpha_smb, alpha_hml])
cov_smb_hml = np.array([
    [cov_smb.loc['const', 'const'], 0],
    [0, cov_hml.loc['const', 'const']]
])

R = np.array([
    [1, 0],
    [0, 1]
])

joint_wald = vector.T @ np.linalg.inv(R @ cov_smb_hml @ R.T) @ vector
p_value_joint = 1 - chi2.cdf(joint_wald, df = 2)

# Save to file

results_3c = pd.DataFrame({
    'Hypothesis': ['alpha_smb = 0', 'alpha_hml = 0', 'alpha_smb = alpha_hml = 0'],
    'Wald Statistic': [wald_alpha_smb, wald_alpha_hml, joint_wald],
    'p-value': [p_value_alpha_smb, p_value_alpha_hml, p_value_joint]
})

with pd.ExcelWriter(output_file, engine='openpyxl', mode='a' if os.path.exists(output_file) else 'w', if_sheet_exists='replace') as writer:
    results_3c.to_excel(writer, sheet_name = '3c)Wald_statistics', index = False)
